In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
BATCH_SIZE = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LEARNING_RATE = 1e-3
EPOCHS = 10
NOISE_FACTOR = 0.5

In [ ]:
train_data = datasets.CIFAR10(root='./data', train=True, transform=transforms.ToTensor(), download=True)
test_data = datasets.CIFAR10(root='./data', train=False, transform=transforms.ToTensor(), download=True)

train_dataloader = DataLoader(dataset= train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
test_dataloader = DataLoader(dataset=test_data, batch_size=10, shuffle=False, pin_memory=True)

In [ ]:
train_data.class_to_idx

In [ ]:
train_data[0][0].shape

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, ):
        super().__init__()

        self.encoder = nn.Sequential(
            # shape -> [3, 32, 32]na
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=(3, 3), stride=2, padding=1),
            nn.BatchNorm2d(num_features=16),
            nn.ReLU(inplace=True),

            # shape -> [16, 16, 16]
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3, 3), stride=2, padding=1),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(inplace=True),

            # shape -> [32, 8, 8]
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3, 3), stride=2, padding=1),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(inplace=True),

            # shape -> [64, 4, 4]
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(3, 3), stride=2, padding=1),
            # shape -> [128, 2, 2]
            # latent dimension -> 128*2*2 = 512
        )

        self.decoder = nn.Sequential(
            # here is the reversed process of the encoder
            nn.ConvTranspose2d(in_channels=128, out_channels=64, kernel_size=(3, 3), stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=(3, 3), stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=(3, 3), stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(num_features=16),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(in_channels=16, out_channels=3, kernel_size=(3, 3), stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)

        return decoded

In [ ]:
model = Autoencoder()
model.to(device=DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

In [ ]:
model.train()
for epoch in range(EPOCHS):
    train_loss = 0.0
    for images, _ in train_dataloader:
        images = images.to(device=DEVICE)
        noisy_images = images + NOISE_FACTOR * torch.randn_like(images)
        noisy_images = torch.clamp(noisy_images, 0., 1.)

        outputs = model(noisy_images)
        loss = criterion(outputs, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    
    print(f"Epoch: {epoch+1}/{EPOCHS}, Loss: {train_loss}")